# Stage 2: Chunking + Embedding (InLegalBERT) on a subset of IN-Abs

Runs on **300 train / 60 validate / 100 test** documents (editable). The same cells later run on the whole dataset: set `SUBSET_COUNTS = None`.

```
processed/<split>/<doc>.json     <- preprocessing (done)
      | Part A: chunking (CPU)
chunks/<split>/<doc>.json        token ids; a sentence is split only if > 510 tokens
      | Part B: embedding (GPU)
embeddings/<split>/<doc>.pt      (num_sentences x 768): one node feature vector per sentence
```

- Only documents that **have a reference summary** are eligible (the oracle labels and evaluation need it).
- The chosen documents are saved in `subset_ids.json`. **Reuse that file in the NER, graph and GNN stages** so every stage uses the same documents.
- Reference summaries are **not** chunked or embedded.
- Resumable: finished documents are skipped, failed ones are retried.
- **Runtime -> Change runtime type -> T4 GPU** before Part B (Part A works on CPU).

In [ ]:
# CELL 1: install (Colab already has torch; no version pinning)
!pip install -q transformers tqdm sentencepiece
print('done')


done


In [ ]:
# CELL 2: mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# CELL 3: CONFIG  <-- edit here
import os, glob, json, time, random
from collections import Counter

PROJECT_DIR   = '/content/drive/MyDrive/inabs_project'      # OUTPUT_DIR of the preprocessing notebook
PROCESSED_DIR = f'{PROJECT_DIR}/processed'
SUMMARIES_DIR = f'{PROJECT_DIR}/summaries'
CHUNKS_DIR    = f'{PROJECT_DIR}/chunks'
EMB_DIR       = f'{PROJECT_DIR}/embeddings'
SUBSET_FILE   = f'{PROJECT_DIR}/subset_ids.json'
SPLITS        = ['train', 'validate', 'test']

# ---- which documents
SUBSET_COUNTS = (300, 60, 100)   # (train, validate, test).  None = the WHOLE dataset
SELECTION     = 'random'         # 'random' (recommended) or 'first' (first N by filename)
SEED          = 42

# ---- chunking
MODEL_NAME = 'law-ai/InLegalBERT'
CHUNK_SIZE = 512      # model limit INCLUDING [CLS]/[SEP] -> at most 510 real tokens per chunk
OVERLAP    = 0        # tokens shared by consecutive chunks of ONE long sentence

# ---- embedding (GPU)
MAX_BATCH_TOKENS = 16384   # padded tokens per forward pass; lower to 8192 if you hit out-of-memory
MAX_BATCH_SEQS   = 256
USE_FP16         = True    # half-precision inference on GPU (about 2x faster, same quality)
SAVE_FP16        = False   # True halves file size (useful for the full dataset); cast back to float32 when training

assert OVERLAP < CHUNK_SIZE - 2
for d in (CHUNKS_DIR, EMB_DIR):
    os.makedirs(d, exist_ok=True)
print('config ok')

config ok


In [ ]:
# CELL 4: choose the documents (saved to subset_ids.json)
def build_subset():
    subset, info = {}, {}
    for i, sp in enumerate(SPLITS):
        pdir = f'{PROCESSED_DIR}/{sp}'
        sdir = f'{SUMMARIES_DIR}/{sp}'
        proc = {f[:-5] for f in os.listdir(pdir) if f.endswith('.json')} if os.path.isdir(pdir) else set()
        summ = {f[:-4] for f in os.listdir(sdir) if f.endswith('.txt')}  if os.path.isdir(sdir) else set()
        eligible = sorted(proc & summ)                    # must have BOTH a processed judgment and a summary
        n = len(eligible) if SUBSET_COUNTS is None else SUBSET_COUNTS[i]
        if SELECTION == 'first':
            chosen = eligible[:n]
        else:
            order = eligible[:]
            random.Random(SEED).shuffle(order)            # same seed -> a bigger subset contains the smaller one
            chosen = sorted(order[:n])
        subset[sp] = chosen
        info[sp] = dict(processed=len(proc), with_summary=len(eligible), without_summary=len(proc - summ), chosen=len(chosen))
        if len(chosen) < n:
            print(f'!! {sp}: asked for {n} but only {len(chosen)} eligible documents exist')
    return subset, info

SUBSET, info = build_subset()
json.dump({'selection': SELECTION, 'seed': SEED, 'counts': SUBSET_COUNTS, 'ids': SUBSET},
          open(SUBSET_FILE, 'w'), indent=1)

print(f'{"split":<9}{"processed":>10}{"with summary":>14}{"no summary":>12}{"CHOSEN":>8}')
for sp in SPLITS:
    r = info[sp]; print(f'{sp:<9}{r["processed"]:>10,}{r["with_summary"]:>14,}{r["without_summary"]:>12,}{r["chosen"]:>8,}')
print(f'\nTotal chosen: {sum(len(v) for v in SUBSET.values())} documents -> saved to {SUBSET_FILE}')

def subset_items():
    return [(sp, d) for sp in SPLITS for d in SUBSET[sp]]

split     processed  with summary  no summary  CHOSEN
train         5,346         5,346           0     300
validate        712           712           0      60
test          1,070         1,070           0     100

Total chosen: 460 documents -> saved to /content/drive/MyDrive/inabs_project/subset_ids.json


## Part A: Chunking (CPU)

In [ ]:
# CELL 5: load the tokenizer (no model, no GPU needed)
import transformers, logging
transformers.logging.set_verbosity_error()
logging.getLogger('transformers').setLevel(logging.ERROR)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('vocab size :', f'{tokenizer.vocab_size:,}')
print('max length :', tokenizer.model_max_length)
print(tokenizer.tokenize('The appellant filed a suit under Section 34 of the Arbitration Act, 1940.')[:12])

vocab size : 30,522
max length : 512
['the', 'appellant', 'filed', 'a', 'suit', 'under', 'section', '34', 'of', 'the', 'arbitration', 'act']


In [ ]:
# CELL 6: chunking functions
def split_ids(token_ids, size, overlap):
    """Split a token-id list into windows of `size`; consecutive windows share `overlap` tokens.
    size=6, overlap=2:  [A B C D E F G H] -> [A B C D E F] + [E F G H]"""
    if len(token_ids) <= size:
        return [token_ids]
    stride, out, start = size - overlap, [], 0
    while start < len(token_ids):
        end = min(start + size, len(token_ids))
        out.append(token_ids[start:end])
        if end == len(token_ids):
            break
        start += stride
    return out


def build_chunks(token_ids, tokenizer, chunk_size, overlap):
    """One sentence -> list of chunk dicts (token_ids already wrapped as [CLS] ... [SEP])."""
    pieces  = split_ids(token_ids, chunk_size - 2, overlap)
    chunked = len(pieces) > 1
    chunks  = []
    for i, p in enumerate(pieces):
        c = {'chunk_id': i,
             'token_ids': [tokenizer.cls_token_id] + p + [tokenizer.sep_token_id],
             'num_tokens': len(p) + 2,
             'is_chunked': chunked}
        if chunked:                                   # readable text only for the rare split sentences
            c['text'] = tokenizer.decode(p, skip_special_tokens=True)
        chunks.append(c)
    return chunks


def chunk_document(doc, tokenizer, chunk_size, overlap, batch=256):
    sents, ids_all = doc['sentences'], []
    for i in range(0, len(sents), batch):             # batch tokenization is much faster than one call per sentence
        enc = tokenizer([s['text'] for s in sents[i:i + batch]], truncation=False, add_special_tokens=False)
        ids_all += enc['input_ids']
    out, total_chunks, n_split = [], 0, 0
    for s, ids in zip(sents, ids_all):
        chunks = build_chunks(ids, tokenizer, chunk_size, overlap)
        total_chunks += len(chunks); n_split += len(chunks) > 1
        out.append({'sent_id': s['id'], 'section': s['section'], 'original_text': s['text'],
                    'num_tokens': len(ids), 'num_chunks': len(chunks), 'chunks': chunks})
    return {'doc_id': doc['doc_id'], 'split': doc['split'], 'filename': doc.get('filename', ''),
            'tokenizer': MODEL_NAME, 'chunk_size': chunk_size, 'overlap': overlap,
            'num_sentences': len(out), 'total_chunks': total_chunks, 'sentences_chunked': n_split,
            'sentences': out}


def chunk_file_is_current(path):
    """True if an existing chunk file was made with the CURRENT tokenizer/chunk settings."""
    try:
        d = json.load(open(path, encoding='utf-8'))
        return (d.get('tokenizer') == MODEL_NAME and d.get('chunk_size') == CHUNK_SIZE and d.get('overlap') == OVERLAP)
    except Exception:
        return False


def run_chunking(items, desc='Chunking'):
    """Resumable. Skips documents already chunked with the same settings; retries failed ones."""
    from tqdm.auto import tqdm
    todo, stale = [], 0
    for sp, did in items:
        out = f'{CHUNKS_DIR}/{sp}/{did}.json'
        if os.path.exists(out):
            if chunk_file_is_current(out): continue
            stale += 1                                 # made with different settings -> redo
        todo.append((sp, did, out))
    print(f'Selected: {len(items):,} | already done: {len(items)-len(todo):,} | to do: {len(todo):,}'
          + (f' (incl. {stale} re-done: settings changed)' if stale else ''))
    c, t0 = Counter(), time.time()
    for sp, did, out in tqdm(todo, desc=desc, unit='doc'):
        try:
            doc = json.load(open(f'{PROCESSED_DIR}/{sp}/{did}.json', encoding='utf-8'))
            res = chunk_document(doc, tokenizer, CHUNK_SIZE, OVERLAP)
            os.makedirs(os.path.dirname(out), exist_ok=True)
            with open(out + '.tmp', 'w', encoding='utf-8') as fh:
                json.dump(res, fh, ensure_ascii=False)
            os.replace(out + '.tmp', out)             # atomic write
            c['ok'] += 1; c['sent'] += res['num_sentences']
            c['chunks'] += res['total_chunks']; c['split'] += res['sentences_chunked']
        except Exception as e:
            c['err'] += 1
            with open(f'{PROJECT_DIR}/chunk_errors.jsonl', 'a') as ef:
                ef.write(json.dumps({'doc_id': did, 'error': repr(e)}) + '\n')
    print(f"DONE ok={c['ok']:,} errors={c['err']:,} time={(time.time()-t0)/60:.1f} min | sentences={c['sent']:,} "
          f"chunks={c['chunks']:,} split={c['split']:,} ({c['split']/max(c['sent'],1)*100:.2f}%)")
    if c['err']: print('errors logged to chunk_errors.jsonl; re-run this cell to retry')

print('chunking functions defined')

chunking functions defined


In [ ]:
# CELL 7: TEST chunking (writes nothing)
fake = tokenizer(' '.join(['appellant'] * 1300), add_special_tokens=False)['input_ids']
ch = build_chunks(fake, tokenizer, CHUNK_SIZE, OVERLAP)
print(f'synthetic sentence: {len(fake)} tokens -> {len(ch)} chunks, sizes {[c["num_tokens"] for c in ch]}')
assert all(c['num_tokens'] <= CHUNK_SIZE for c in ch)
if OVERLAP:
    assert ch[0]['token_ids'][-1 - OVERLAP:-1] == ch[1]['token_ids'][1:1 + OVERLAP], 'overlap broken'
else:
    assert sum(c['num_tokens'] - 2 for c in ch) == len(fake), 'tokens lost or duplicated'
print('splitting logic OK\n')

for sp, did in subset_items()[:3]:
    doc = json.load(open(f'{PROCESSED_DIR}/{sp}/{did}.json', encoding='utf-8'))
    res = chunk_document(doc, tokenizer, CHUNK_SIZE, OVERLAP)
    assert [s['sent_id'] for s in res['sentences']] == [s['id'] for s in doc['sentences']]
    print(f"{did[:50]:<50} {res['num_sentences']:>4} sentences -> {res['total_chunks']:>4} chunks ({res['sentences_chunked']} split)")
print('\ntest passed')

synthetic sentence: 1300 tokens -> 3 chunks, sizes [512, 512, 282]
splitting logic OK

train_1007                                           42 sentences ->   44 chunks (2 split)
train_1035                                           58 sentences ->   58 chunks (0 split)
train_1037                                           69 sentences ->   69 chunks (0 split)

test passed


In [ ]:
# CELL 8: RUN CHUNKING on the chosen documents (resumable: just re-run after a disconnect)
run_chunking(subset_items())

Selected: 460 | already done: 460 | to do: 0


Chunking: 0doc [00:00, ?doc/s]

DONE ok=0 errors=0 time=0.0 min | sentences=0 chunks=0 split=0 (0.00%)


In [ ]:
# CELL 9: VERIFY chunks
import statistics as st
rows = []
for sp, did in subset_items():
    f = f'{CHUNKS_DIR}/{sp}/{did}.json'
    assert os.path.exists(f), f'missing chunk file: {did}'
    d   = json.load(open(f, encoding='utf-8'))
    src = json.load(open(f'{PROCESSED_DIR}/{sp}/{did}.json', encoding='utf-8'))
    assert [s['sent_id'] for s in d['sentences']] == [s['id'] for s in src['sentences']], f'ALIGNMENT BUG: {did}'
    assert all(c['num_tokens'] <= CHUNK_SIZE for s in d['sentences'] for c in s['chunks']), f'chunk too long: {did}'
    rows.append((sp, d))
print(f'{len(rows)} chunk files OK: ids align with processed/, no chunk exceeds {CHUNK_SIZE} tokens\n')
print(f'{"split":<9}{"docs":>6}{"sentences":>11}{"chunks":>9}{"split_sent":>12}')
for sp in SPLITS:
    r = [d for s, d in rows if s == sp]
    if r: print(f'{sp:<9}{len(r):>6}{sum(d["num_sentences"] for d in r):>11,}{sum(d["total_chunks"] for d in r):>9,}{sum(d["sentences_chunked"] for d in r):>12,}')
tk = [s['num_tokens'] for _, d in rows for s in d['sentences']]; q = st.quantiles(tk, n=100)
print(f'\ntokens/sentence: median {st.median(tk):.0f} | p90 {q[89]:.0f} | p99 {q[98]:.0f} | max {max(tk):,}')
print('(share of sentences > 510 tokens should be tiny; if not, sentence splitting is failing)')

460 chunk files OK: ids align with processed/, no chunk exceeds 512 tokens

split      docs  sentences   chunks  split_sent
train       300     28,387   28,942         330
validate     60      5,023    5,140          54
test        100      8,700    8,850          87

tokens/sentence: median 32 | p90 79 | p99 560 | max 6,911
(share of sentences > 510 tokens should be tiny; if not, sentence splitting is failing)


## Part B: Embedding (GPU: Runtime -> Change runtime type -> T4)

In [ ]:
# CELL 10: load InLegalBERT
import torch
import torch.nn.functional as F
from transformers import AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    print('GPU :', torch.cuda.get_device_name(0), f'| VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('!! No GPU: embedding will be very slow. Runtime -> Change runtime type -> T4 GPU')

model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
EMB_DIM = model.config.hidden_size
MAX_POS = model.config.max_position_embeddings
print(f'model loaded | embedding dim = {EMB_DIM} | max positions = {MAX_POS}')
assert CHUNK_SIZE <= MAX_POS

device: cpu
!! No GPU: embedding will be very slow. Runtime -> Change runtime type -> T4 GPU


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model loaded | embedding dim = 768 | max positions = 512


In [ ]:
# CELL 11: embedding functions
def make_batches(lengths, max_tokens, max_seqs):
    """Group chunk indices by length so little padding is wasted.
    Each batch holds at most max_seqs chunks and at most max_tokens padded tokens."""
    order = sorted(range(len(lengths)), key=lambda i: lengths[i])
    batches, cur = [], []
    for i in order:
        L = lengths[i]                                 # ascending order -> L is the batch maximum so far
        if cur and (len(cur) + 1 > max_seqs or (len(cur) + 1) * L > max_tokens):
            batches.append(cur); cur = []
        cur.append(i)
    if cur: batches.append(cur)
    return batches


@torch.no_grad()
def encode_chunks(token_lists, max_tokens, max_seqs):
    """token_lists: list of id lists (already [CLS]...[SEP]). Returns (n, EMB_DIM) float32 on CPU.
    Mean-pools the last hidden state over real (non-padding) tokens, same pooling as your earlier notebook."""
    lengths = [len(t) for t in token_lists]
    out = torch.zeros(len(token_lists), EMB_DIM)
    use_amp = USE_FP16 and device.type == 'cuda'
    for b in make_batches(lengths, max_tokens, max_seqs):
        L    = max(lengths[i] for i in b)
        ids  = torch.full((len(b), L), tokenizer.pad_token_id, dtype=torch.long)
        mask = torch.zeros((len(b), L), dtype=torch.long)
        for r, i in enumerate(b):
            t = token_lists[i]
            ids[r, :len(t)] = torch.tensor(t); mask[r, :len(t)] = 1
        ids, mask = ids.to(device), mask.to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            h = model(input_ids=ids, attention_mask=mask).last_hidden_state
        h = h.float(); m = mask.unsqueeze(-1).float()
        out[torch.tensor(b)] = ((h * m).sum(1) / m.sum(1).clamp(min=1)).cpu()
    return out


def embed_document(chunk_doc):
    """One chunk file -> dict with a (num_sentences, EMB_DIM) tensor. Sentences split into several chunks are
    averaged back to ONE vector, weighted by each chunk's token count."""
    sents = chunk_doc['sentences']
    if not sents: return None
    owner, toks, w = [], [], []
    for si, s in enumerate(sents):
        for c in s['chunks']:
            assert c['num_tokens'] <= MAX_POS
            owner.append(si); toks.append(c['token_ids']); w.append(max(c['num_tokens'] - 2, 1))

    budget = MAX_BATCH_TOKENS
    while True:                                        # on out-of-memory: halve the batch budget and retry
        try:
            emb = encode_chunks(toks, budget, MAX_BATCH_SEQS); break
        except RuntimeError as e:
            if 'out of memory' not in str(e).lower() or budget <= 512: raise
            torch.cuda.empty_cache(); budget //= 2
            print(f'  OOM -> retrying with MAX_BATCH_TOKENS={budget}')

    n   = len(sents)
    own = torch.tensor(owner); wt = torch.tensor(w, dtype=torch.float32)
    num = torch.zeros(n, EMB_DIM).index_add_(0, own, emb * wt[:, None])
    den = torch.zeros(n).index_add_(0, own, wt)
    sent_emb = num / den[:, None]
    if not torch.isfinite(sent_emb).all():
        raise ValueError('non-finite values in embeddings (try USE_FP16 = False)')
    return {'doc_id': chunk_doc['doc_id'], 'split': chunk_doc['split'], 'filename': chunk_doc.get('filename', ''),
            'num_sentences': n,
            'embeddings': sent_emb.half() if SAVE_FP16 else sent_emb,
            'sent_ids':   [s['sent_id'] for s in sents],
            'sections':   [s['section'] for s in sents],
            'num_tokens': [s['num_tokens'] for s in sents],
            'num_chunks': [s['num_chunks'] for s in sents],
            'model': MODEL_NAME}


def run_embedding(items, desc='Embedding'):
    """Resumable: skips documents whose .pt already exists; failed documents are retried on the next run."""
    from tqdm.auto import tqdm
    todo = [(sp, d) for sp, d in items if not os.path.exists(f'{EMB_DIR}/{sp}/{d}.pt')]
    print(f'Selected: {len(items):,} | already done: {len(items)-len(todo):,} | to do: {len(todo):,}')
    c, t0 = Counter(), time.time()
    for sp, did in tqdm(todo, desc=desc, unit='doc'):
        try:
            cd = json.load(open(f'{CHUNKS_DIR}/{sp}/{did}.json', encoding='utf-8'))
            if cd.get('tokenizer') != MODEL_NAME or cd.get('chunk_size') != CHUNK_SIZE:
                raise ValueError('chunk file made with other settings: re-run Part A (CELL 8)')
            res = embed_document(cd)
            if res is None: c['empty'] += 1; continue
            out = f'{EMB_DIR}/{sp}/{did}.pt'
            os.makedirs(os.path.dirname(out), exist_ok=True)
            torch.save(res, out + '.tmp'); os.replace(out + '.tmp', out)     # atomic write
            c['ok'] += 1; c['sent'] += res['num_sentences']
        except Exception as e:
            c['err'] += 1
            with open(f'{PROJECT_DIR}/embed_errors.jsonl', 'a') as ef:
                ef.write(json.dumps({'doc_id': did, 'error': repr(e)}) + '\n')
            print(f'  ERROR {did}: {e}')
    el = time.time() - t0
    print(f"DONE ok={c['ok']:,} empty={c['empty']} errors={c['err']} | {c['sent']:,} sentence vectors | "
          f"{el/60:.1f} min ({c['ok']/max(el,1e-9):.2f} doc/s)")
    if c['err']: print('errors logged to embed_errors.jsonl; re-run this cell to retry')

print('embedding functions defined')

embedding functions defined


In [ ]:
# CELL 12: SELF-TEST. Batched embedding must equal embedding each sentence on its own (writes nothing)
# 1) batching logic: every index appears exactly once, and no batch exceeds its budget
lens = [random.randint(5, 500) for _ in range(1000)]
bs = make_batches(lens, 4096, 64)
assert sorted(i for b in bs for i in b) == list(range(1000))
assert all(len(b) <= 64 and len(b) * max(lens[i] for i in b) <= 4096 or len(b) == 1 for b in bs)
print('batching logic OK:', len(bs), 'batches for 1000 chunks')

# 2) numerical check on real sentences (only single-chunk sentences are comparable)
sp, did = subset_items()[0]
cd = json.load(open(f'{CHUNKS_DIR}/{sp}/{did}.json', encoding='utf-8'))
sample = [s for s in cd['sentences'] if s['num_chunks'] == 1][:8]
res = embed_document({**cd, 'sentences': sample})
sims = []
for s, v in zip(sample, res['embeddings'].float()):
    enc = tokenizer(s['original_text'], return_tensors='pt', truncation=True, max_length=CHUNK_SIZE).to(device)
    with torch.no_grad():
        direct = model(**enc).last_hidden_state[0].float().mean(0).cpu()
    sims.append(F.cosine_similarity(direct, v, dim=0).item())
print('cosine(batched, one-by-one):', [round(x, 5) for x in sims])
assert min(sims) > 0.995, 'batched embeddings differ from single embeddings: padding/mask bug'
print('numerical check OK\n')

res = embed_document(cd)
print(f"{did[:50]}: {res['num_sentences']} sentences -> embeddings {tuple(res['embeddings'].shape)}, dtype {res['embeddings'].dtype}")
assert res['embeddings'].shape == (len(cd['sentences']), EMB_DIM)
print('self-test passed')

batching logic OK: 68 batches for 1000 chunks
cosine(batched, one-by-one): [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
numerical check OK

train_1007: 42 sentences -> embeddings (42, 768), dtype torch.float32
self-test passed


In [14]:
# CELL 13: RUN EMBEDDING on the chosen documents (resumable: just re-run after a disconnect)
run_embedding(subset_items())

Selected: 460 | already done: 80 | to do: 380


Embedding:   0%|          | 0/380 [00:00<?, ?doc/s]

DONE ok=380 empty=0 errors=0 | 35,480 sentence vectors | 305.9 min (0.02 doc/s)


In [15]:
# CELL 14: VERIFY embeddings
rows, bad = [], 0
for sp, did in subset_items():
    f = f'{EMB_DIR}/{sp}/{did}.pt'
    if not os.path.exists(f): bad += 1; continue
    e   = torch.load(f, map_location='cpu')
    src = json.load(open(f'{PROCESSED_DIR}/{sp}/{did}.json', encoding='utf-8'))
    assert e['embeddings'].shape == (len(src['sentences']), EMB_DIM), f'ROW MISMATCH: {did}'
    assert e['sent_ids'] == [s['id'] for s in src['sentences']], f'ID MISMATCH: {did}'
    assert torch.isfinite(e['embeddings'].float()).all(), f'NaN/inf: {did}'
    rows.append((sp, e))
print(f'{len(rows)} embedding files OK (rows == sentences, ids align with processed/, all finite)'
      + (f'  |  {bad} MISSING: re-run CELL 13' if bad else ''))

print(f'\n{"split":<9}{"docs":>6}{"vectors":>10}{"min":>6}{"median":>8}{"max":>7}')
for sp in SPLITS:
    n = [e['num_sentences'] for s, e in rows if s == sp]
    if n: print(f'{sp:<9}{len(n):>6}{sum(n):>10,}{min(n):>6}{sorted(n)[len(n)//2]:>8}{max(n):>7}')
tot = sum(e['num_sentences'] for _, e in rows)
print(f'\ntotal vectors: {tot:,} | storage ~ {tot*EMB_DIM*(2 if SAVE_FP16 else 4)/1e6:.0f} MB')

# collapse check: embeddings of different sentences must not all be identical
e = max((e for _, e in rows), key=lambda x: x['num_sentences'])['embeddings'].float()
g = torch.Generator().manual_seed(0)
a, b = torch.randint(0, len(e), (2000,), generator=g), torch.randint(0, len(e), (2000,), generator=g)
keep = a != b
print(f'mean cosine between random sentence pairs: {F.cosine_similarity(e[a[keep]], e[b[keep]], dim=1).mean():.3f}')
print('BERT-style embeddings are anisotropic: expect a HIGH value (0.8-0.95). It is not a bug, but it means that for')
print('the graph step you should use top-k neighbours (not a fixed threshold) and mean-center the vectors first.')

460 embedding files OK (rows == sentences, ids align with processed/, all finite)

split      docs   vectors   min  median    max
train       300    28,387     6      67   1587
validate     60     5,023    15      61    482
test        100     8,700     9      68    672

total vectors: 42,110 | storage ~ 129 MB
mean cosine between random sentence pairs: 0.757
BERT-style embeddings are anisotropic: expect a HIGH value (0.8-0.95). It is not a bug, but it means that for
the graph step you should use top-k neighbours (not a fixed threshold) and mean-center the vectors first.


## Next steps
1. **Scaling to the whole dataset:** set `SUBSET_COUNTS = None` in Cell 3 and re-run Cells 4, 8, 13 (and 9, 14 to verify). Finished documents are skipped.
2. **NER, graph, oracle labels and GNN** read `subset_ids.json` so they use exactly these documents.
3. Keep `embeddings/` and `chunks/` in Drive; they are inputs to the graph stage.